In [2]:
from __future__ import annotations

import requests
import pandas as pd
from typing import Dict
from datetime import datetime, timedelta, timezone

In [ ]:
KST = timezone(timedelta(hours=9))
PROMETHEUS_URL = "http://10.101.5.160:9010"

FEATURE_QUERIES: Dict[str, str] = {
    "http_error_rate": """
        (
            sum(rate(http_server_requests_seconds_count{status=~"5.."}[5m]))
            or vector(0)
        )
        /
        clamp_min(sum(rate(http_server_requests_seconds_count[5m])), 1e-9)
    """,
    "latency_p95": """
        histogram_quantile(
            0.95,
            sum by (le) (rate(http_server_requests_seconds_bucket[5m]))
        ) * 1000
    """,
    "latency_p99": """
        histogram_quantile(
            0.99,
            sum by (le) (rate(http_server_requests_seconds_bucket[5m]))
        ) * 1000
    """,
    "throughput": """
        sum(rate(http_server_requests_seconds_count[5m]))
    """,
    "service_status": """
        min(up)
    """,
    "cpu_pressure": """
        (
            clamp_max(avg(system_cpu_usage), 1) * 0.7
        )
        +
        (
            clamp_max(avg(system_load_average_1m) / clamp_min(avg(system_cpu_count), 1), 1) * 0.3
        )
    """,
    "memory_pressure": """
        (
            clamp_max(
                1 - (
                    sum(node_memory_MemAvailable_bytes)
                    /
                    clamp_min(sum(node_memory_MemTotal_bytes), 1)
                ),
                1
            ) * 0.5
        )
        +
        (
            clamp_max(
                sum(jvm_memory_used_bytes{area="heap"})
                /
                clamp_min(sum(jvm_memory_max_bytes{area="heap"}), 1),
                1
            ) * 0.5
        )
    """,
    "db_pool_pressure": """
        (
            (
                sum(hikaricp_connections_active)
                /
                clamp_min(sum(hikaricp_connections_max), 1)
            ) * 0.6
        )
        +
        (
            clamp_max(sum(hikaricp_connections_pending) / 10, 1) * 0.3
        )
        +
        (
            clamp_max(sum(rate(hikaricp_connections_timeout_total[5m])) / 1, 1) * 0.1
        )
    """,
    "web_pressure": """
        (
            clamp_max(
                sum(tomcat_threads_busy_threads)
                /
                clamp_min(sum(tomcat_threads_config_max_threads), 1),
                1
            ) * 0.7
        )
        +
        (
            clamp_max(
                sum(tomcat_connections_current_connections)
                /
                clamp_min(sum(tomcat_connections_config_max_connections), 1),
                1
            ) * 0.3
        )
    """,
    "disk_pressure": """
        (
            clamp_max(
                1 - (
                    sum(node_filesystem_free_bytes{fstype!~"tmpfs|overlay", mountpoint="/"})
                    /
                    clamp_min(sum(node_filesystem_size_bytes{fstype!~"tmpfs|overlay", mountpoint="/"}), 1)
                ),
                1
            ) * 0.8
        )
        +
        (
            clamp_max(sum(node_disk_io_now) / 10, 1) * 0.2
        )
    """,
    "network_pressure": """
        (
            clamp_max(sum(rate(node_netstat_Tcp_RetransSegs[5m])) / 1, 1) * 0.5
        )
        +
        (
            clamp_max(sum(rate(node_network_receive_drop_total[5m])) / 0.1, 1) * 0.25
        )
        +
        (
            clamp_max(sum(rate(node_network_transmit_drop_total[5m])) / 0.1, 1) * 0.25
        )
    """,
}


class PrometheusClient:
    def __init__(self, base_url: str, timeout: int = 30) -> None:
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.session = requests.Session()

    def query_range(
        self,
        query: str,
        start: datetime,
        end: datetime,
        step: str = "60s",
    ) -> dict:
        url = f"{self.base_url}/api/v1/query_range"
        params = {
            "query": query,
            "start": start.timestamp(),
            "end": end.timestamp(),
            "step": step,
        }
        resp = self.session.get(url, params=params, timeout=self.timeout)
        resp.raise_for_status()

        payload = resp.json()
        if payload.get("status") != "success":
            raise RuntimeError(f"Prometheus query failed: {payload}")

        return payload["data"]

    def fetch_series_as_df(
        self,
        name: str,
        query: str,
        start: datetime,
        end: datetime,
        step: str = "60s",
    ) -> pd.DataFrame:
        data = self.query_range(query=query, start=start, end=end, step=step)
        result = data.get("result", [])

        if not result:
            return pd.DataFrame(columns=["timestamp", name])

        values = result[0].get("values", [])

        rows = []
        for ts, value in values:
            rows.append(
                {
                    "timestamp": pd.to_datetime(float(ts), unit="s", utc=True).tz_convert(KST),
                    name: None if value in ("NaN", "nan", "null", None) else float(value),
                }
            )

        return pd.DataFrame(rows)


def build_feature_dataframe(
    client: PrometheusClient,
    queries: Dict[str, str],
    start: datetime,
    end: datetime,
    step: str = "60s",
) -> pd.DataFrame:
    merged_df: pd.DataFrame | None = None

    for feature_name, promql in queries.items():
        feature_df = client.fetch_series_as_df(
            name=feature_name,
            query=promql,
            start=start,
            end=end,
            step=step,
        )

        if merged_df is None:
            merged_df = feature_df
        else:
            merged_df = merged_df.merge(feature_df, on="timestamp", how="outer")

    if merged_df is None:
        return pd.DataFrame()

    merged_df = merged_df.sort_values("timestamp").reset_index(drop=True)

    feature_cols = [c for c in merged_df.columns if c != "timestamp"]
    for col in feature_cols:
        merged_df[col] = pd.to_numeric(merged_df[col], errors="coerce")

    merged_df[feature_cols] = merged_df[feature_cols].ffill().bfill()

    merged_df[feature_cols] = merged_df[feature_cols].fillna(0.0)

    return merged_df


def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    out = df.copy()

    delta_targets = [
        "http_error_rate",
        "latency_p95",
        "throughput",
        "cpu_pressure",
        "memory_pressure",
    ]

    for col in delta_targets:
        if col in out.columns:
            out[f"{col}_delta"] = out[col].diff().fillna(0.0)

    return out


def filter_normal_candidates(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    conditions = (
        (df["service_status"] >= 1.0)
        & (df["http_error_rate"] < 0.05)
        & (df["latency_p95"] < df["latency_p95"].quantile(0.95))
        & (df["throughput"] > 0)
    )

    return df.loc[conditions].reset_index(drop=True)


def get_prometheus_feature_dataset(
    prometheus_url: str,
    hours: int = 24,
    step: str = "60s",
    add_derived: bool = True,
    normal_only: bool = False,
) -> pd.DataFrame:
    client = PrometheusClient(prometheus_url)

    end = datetime.now(tz=KST)
    start = end - timedelta(hours=hours)

    df = build_feature_dataframe(
        client=client,
        queries=FEATURE_QUERIES,
        start=start,
        end=end,
        step=step,
    )

    if add_derived:
        df = add_derived_features(df)

    if normal_only:
        df = filter_normal_candidates(df)

    return df

def filter_business_hours(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    df = df.copy()

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    df = df.dropna(subset=["timestamp"])

    df["hour"] = df["timestamp"].dt.hour

    filtered = df[(df["hour"] >= 10) & (df["hour"] < 22)].copy()

    return filtered.drop(columns=["hour"])


if __name__ == "__main__":
    df = get_prometheus_feature_dataset(
        prometheus_url=PROMETHEUS_URL,
        hours=24 * 7,
        step="60s",
        add_derived=True,
        normal_only=False,
    )
    
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    df = df[df["service_status"] > 0]
    
    target_date = pd.to_datetime("2026-04-07")

    df = df[df['timestamp'].dt.date == target_date.date()]
    
    df = filter_business_hours(df)
    
    # df = df[df["timestamp"].dt.weekday < 5]
    print(df.head())
    print(df.columns.tolist())
    print(df.shape)
    

    df.to_csv("prometheus_feature_dataset.csv", index=False, encoding="utf-8-sig")

                              timestamp  http_error_rate  latency_p95  \
300 2026-04-01 16:26:53.551000118+09:00              0.0   214.989060   
301 2026-04-01 16:27:53.551000118+09:00              0.0   214.989060   
302 2026-04-01 16:28:53.551000118+09:00              0.0   206.041211   
303 2026-04-01 16:29:53.551000118+09:00              0.0    33.298967   
304 2026-04-01 16:30:53.551000118+09:00              0.0   290.805077   

     latency_p99  throughput  service_status  cpu_pressure  memory_pressure  \
300   221.954781    0.012975             1.0      0.812500         0.587300   
301   221.954781    0.012975             1.0      0.152181         0.693438   
302   220.165211    0.026308             1.0      0.032606         0.696551   
303   218.375641    0.039641             1.0      0.019468         0.700460   
304   344.492168    0.052975             1.0      0.043322         0.705578   

     db_pool_pressure  web_pressure  disk_pressure  network_pressure  \
300           

In [10]:
df

,timestamp,http_error_rate,latency_p95,latency_p99,throughput,service_status,cpu_pressure,memory_pressure,db_pool_pressure,web_pressure,disk_pressure,network_pressure,http_error_rate_delta,latency_p95_delta,throughput_delta,cpu_pressure_delta,memory_pressure_delta
337,2026-04-01 16:27:25.131000042+09:00,0.0,219.227357,222.802440,0.006659,1.0,0.099232,0.613256,0.0,0.00144,0.696188,0.028070,0.0,0.000000,0.000000,0.000000,0.000000
338,2026-04-01 16:28:25.131000042+09:00,0.0,210.279509,221.012871,0.019992,1.0,0.052008,0.694687,0.0,0.00144,0.696186,0.038596,0.0,-8.947848,0.013333,-0.047224,0.081431
339,2026-04-01 16:29:25.131000042+09:00,0.0,201.331660,219.223301,0.033326,1.0,0.024219,0.698585,0.0,0.00144,0.696184,0.036842,0.0,-8.947848,0.013333,-0.027789,0.003898
340,2026-04-01 16:30:25.131000042+09:00,0.0,299.752926,346.281738,0.046659,1.0,0.025074,0.701996,0.0,0.00144,0.696186,0.040351,0.0,98.421265,0.013333,0.000855,0.003411
341,2026-04-01 16:31:25.131000042+09:00,0.0,281.857229,342.702599,0.059992,1.0,0.014151,0.707079,0.0,0.00144,0.696240,0.036842,0.0,-17.895697,0.013333,-0.010923,0.005083
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9305,2026-04-07 21:55:25.131000042+09:00,0.0,3.462046,3.488610,0.066667,1.0,0.017333,0.584692,0.0,0.00144,0.701790,0.022807,0.0,-0.005534,0.000000,-0.001627,0.001802
9306,2026-04-07 21:56:25.131000042+09:00,0.0,3.512727,3.778366,0.066667,1.0,0.012615,0.587584,0.0,0.00144,0.701788,0.021053,0.0,0.050681,0.000000,-0.004718,0.002893
9307,2026-04-07 21:57:25.131000042+09:00,0.0,3.512727,3.778366,0.066667,1.0,0.016992,0.590572,0.0,0.00144,0.701738,0.022807,0.0,0.000000,0.000000,0.004378,0.002988
9308,2026-04-07 21:58:25.131000042+09:00,0.0,3.512727,3.778366,0.066667,1.0,0.010880,0.593536,0.0,0.00144,0.701789,0.021053,0.0,0.000000,0.000000,-0.006112,0.002964


,timestamp,http_error_rate,latency_p95,latency_p99,throughput,service_status,cpu_pressure,memory_pressure,db_pool_pressure,web_pressure,disk_pressure,network_pressure,http_error_rate_delta,latency_p95_delta,throughput_delta,cpu_pressure_delta,memory_pressure_delta
8554,2026-04-07 10:00:53.551000118+09:00,0.0,12.652815,13.715372,0.066667,1.0,0.052836,0.615414,0.0,0.00144,0.699583,0.024561,0.0,0.000000,0.0,0.041146,0.004593
8555,2026-04-07 10:01:53.551000118+09:00,0.0,22.649241,26.899469,0.066667,1.0,0.168122,0.622891,0.0,0.00144,0.699581,0.047368,0.0,9.996426,0.0,0.115286,0.007477
8556,2026-04-07 10:02:53.551000118+09:00,0.0,22.649241,26.899469,0.066667,1.0,0.111877,0.627539,0.0,0.00144,0.699581,0.057895,0.0,0.000000,0.0,-0.056245,0.004649
8557,2026-04-07 10:03:53.551000118+09:00,0.0,22.649241,26.899469,0.066667,1.0,0.200874,0.638754,0.0,0.00144,0.699533,0.057895,0.0,0.000000,0.0,0.088998,0.011214
8558,2026-04-07 10:04:53.551000118+09:00,0.0,22.649241,26.899469,0.066667,1.0,0.123903,0.639389,0.0,0.00144,0.699534,0.087719,0.0,0.000000,0.0,-0.076971,0.000636
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9269,2026-04-07 21:55:53.551000118+09:00,0.0,3.512727,3.778366,0.066667,1.0,0.015503,0.586743,0.0,0.00144,0.701788,0.021053,0.0,0.045147,0.0,-0.008308,0.003986
9270,2026-04-07 21:56:53.551000118+09:00,0.0,3.512727,3.778366,0.066667,1.0,0.021269,0.588590,0.0,0.00144,0.701740,0.021053,0.0,0.000000,0.0,0.005766,0.001847
9271,2026-04-07 21:57:53.551000118+09:00,0.0,3.512727,3.778366,0.066667,1.0,0.013762,0.591637,0.0,0.00144,0.701787,0.021053,0.0,0.000000,0.0,-0.007507,0.003047
9272,2026-04-07 21:58:53.551000118+09:00,0.0,3.512727,3.778366,0.066667,1.0,0.012960,0.594600,0.0,0.00144,0.701787,0.021053,0.0,0.000000,0.0,-0.000802,0.002963


In [5]:
df["timestamp"].dt.date.unique()

array([datetime.date(2026, 4, 1), datetime.date(2026, 4, 2),
       datetime.date(2026, 4, 3), datetime.date(2026, 4, 7)], dtype=object)

In [8]:
import os
import json
import re

TARGET_DIR = "/home/axtft/projects/axtft/data/report/interval/2026/04/03"  # 폴더 경로

pattern = re.compile(r"^(\d{4})_(\d{4})\.json$")

records = []

for filename in os.listdir(TARGET_DIR):
    match = pattern.match(filename)
    if not match:
        continue

    start_time = match.group(1)  # "0000"

    filepath = os.path.join(TARGET_DIR, filename)

    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        if data.get("level") != "high":
            records.append({
                "filename": filename,
                "start_time": int(start_time),
                "data": data
            })

    except Exception as e:
        print(f"ERROR: {filename}, {e}")

# 🔥 시간 기준 정렬
records = sorted(records, key=lambda x: x["start_time"])

# 결과 확인
for r in records:
    print(r["filename"])

0800_0810.json
0810_0820.json
0820_0830.json
0830_0840.json
0840_0850.json
0850_0900.json
0900_0910.json
0910_0920.json
0920_0930.json
0930_0940.json
0940_0950.json
0950_1000.json
1000_1010.json
1010_1020.json
1020_1030.json
1030_1040.json
1050_1100.json
1100_1110.json
1110_1120.json
1120_1130.json
1130_1140.json
1140_1150.json
1150_1200.json
1200_1210.json
1210_1220.json
1220_1230.json
1230_1240.json
1240_1250.json
1250_1300.json
1300_1310.json
1310_1320.json
1330_1340.json
1340_1350.json
1350_1400.json
1400_1410.json
1410_1420.json
1420_1430.json
1430_1440.json
1450_1500.json
1500_1510.json
1510_1520.json
1520_1530.json
1530_1540.json
1540_1550.json
1550_1600.json
1600_1610.json
1610_1620.json
1620_1630.json
1630_1640.json
1640_1650.json
1650_1700.json
1700_1710.json
1710_1720.json
1730_1740.json
1740_1750.json
1750_1800.json
1800_1810.json
1810_1820.json
1820_1830.json
1840_1850.json
1850_1900.json
1900_1910.json
1910_1920.json
1920_1930.json
1930_1940.json
1940_1950.json
1950_2000.

In [7]:
low_files

NameError: name 'low_files' is not defined